In [2]:
import pandas as pd
import numpy as np
import pickle
import os
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.sparse import save_npz, load_npz

# 1. Nạp dữ liệu DataFrame bài báo
data_path = 'data/news_dataset_df_prep2.pkl' 
print("Đang tải dữ liệu bài báo...")
df_load = pd.read_pickle(data_path)
print(f"-> Tải thành công {len(df_load)} bài báo.")

# 2. Nạp Inverted Index (File này bạn lưu ở thư mục gốc CUOI_KY)
print("Đang tải Inverted Index...")
with open("data/inverted_index.pkl", "rb") as f:
    inverted_index = pickle.load(f)
print(f"-> Tải thành công Inverted Index với {len(inverted_index)} từ khóa.")

# 3. Tạo bộ ánh xạ nhanh: từ bài báo ID -> vị trí dòng (index) trong ma trận TF-IDF
print("Đang tạo bộ ánh xạ vị trí...")
doc_id_to_idx = {doc_id: idx for idx, doc_id in enumerate(df_load['id'])}
print("-> Đã chuẩn bị xong bộ lọc Inverted Index và bộ ánh xạ.")

Đang tải dữ liệu bài báo...
-> Tải thành công 160254 bài báo.
Đang tải Inverted Index...
-> Tải thành công Inverted Index với 519282 từ khóa.
Đang tạo bộ ánh xạ vị trí...
-> Đã chuẩn bị xong bộ lọc Inverted Index và bộ ánh xạ.


In [3]:
print("Đang xây dựng ma trận TF-IDF... (Có thể mất 1-2 phút)")

# Tối ưu hóa: Loại bỏ từ quá hiếm (<5 văn bản) hoặc quá phổ biến (>80% văn bản)
vectorizer = TfidfVectorizer(
    min_df=5, 
    max_df=0.8, 
    max_features=100000, 
    lowercase=True
)

# Huấn luyện (Fit) và biến đổi dữ liệu thành ma trận
tfidf_matrix = vectorizer.fit_transform(df_load['combined_processed'].astype(str))

print(f"Kích thước từ vựng (Vocabulary): {len(vectorizer.vocabulary_)}")
print(f"Kích thước ma trận TF-IDF: {tfidf_matrix.shape}")

Đang xây dựng ma trận TF-IDF... (Có thể mất 1-2 phút)
Kích thước từ vựng (Vocabulary): 80106
Kích thước ma trận TF-IDF: (160254, 80106)


In [4]:
def search_documents_inverted(query_text, k, vectorizer, tfidf_matrix, df_documents, inverted_index, doc_id_to_idx):
    print(f"\n**Truy vấn:** '{query_text}'")
    
    # GIAI ĐOẠN 1: LỌC THÔ (Retrieval) dùng Inverted Index
    query_tokens = vectorizer.build_analyzer()(query_text)
    
    candidate_doc_ids = set()
    for token in query_tokens:
        if token in inverted_index:
            candidate_doc_ids.update(inverted_index[token])
            
    print(f"-> Bộ lọc Mục lục tìm thấy: {len(candidate_doc_ids)} văn bản ứng viên (Thay vì phải quét {len(df_documents)} bài).")
    
    if not candidate_doc_ids:
        print("Không tìm thấy tài liệu nào chứa từ khóa từ truy vấn.")
        return []
        
    # Chuyển đổi ID ứng viên sang vị trí index tương ứng trong ma trận
    candidate_indices = [doc_id_to_idx[d_id] for d_id in candidate_doc_ids if d_id in doc_id_to_idx]
    
    # GIAI ĐOẠN 2: XẾP HẠNG CHÍNH XÁC (Ranking) dùng TF-IDF
    # Vector hóa câu truy vấn
    query_vector = vectorizer.transform([query_text])
    
    # Chỉ cắt ma trận TF-IDF tại đúng các dòng của văn bản ứng viên
    sub_tfidf_matrix = tfidf_matrix[candidate_indices]
    
    # Tính Cosine Similarity trên tập ứng viên siêu nhỏ
    similarities = cosine_similarity(query_vector, sub_tfidf_matrix).flatten()
    
    # Lấy Top K kết quả
    top_k_sub_indices = similarities.argsort()[-k:][::-1]
    
    print(f"\n--- Top {k} Tài liệu tương đồng nhất ---")
    results = []
    for rank, sub_idx in enumerate(top_k_sub_indices):
        score = similarities[sub_idx]
        if score <= 0:
            continue
            
        original_idx = candidate_indices[sub_idx]
        
        doc_id = df_documents.iloc[original_idx]['id']
        doc_title = df_documents.iloc[original_idx]['title']
        doc_content = df_documents.iloc[original_idx]['combined_processed']
        
        print(f"\nRank {rank + 1} (Similarity Score: {score:.4f}):")
        print(f"  ID: {doc_id} | Title: {doc_title}")
        print(f"  Content snippet: {str(doc_content)[:150]}...")
        
        results.append({
            'rank': rank + 1,
            'similarity_score': score,
            'doc_id': doc_id,
            'title': doc_title
        })
        
    return results

In [5]:
# Chạy thử truy vấn đầu tiên
query_1 = "cướp tiệm vàng"
k_results = 5

results_1 = search_documents_inverted(
    query_text=query_1, 
    k=k_results, 
    vectorizer=vectorizer, 
    tfidf_matrix=tfidf_matrix, 
    df_documents=df_load, 
    inverted_index=inverted_index, 
    doc_id_to_idx=doc_id_to_idx
)

print("\n" + "="*50)

# Chạy thử truy vấn thứ hai
query_2 = "bóng đá việt nam"
results_2 = search_documents_inverted(
    query_text=query_2, 
    k=k_results, 
    vectorizer=vectorizer, 
    tfidf_matrix=tfidf_matrix, 
    df_documents=df_load, 
    inverted_index=inverted_index, 
    doc_id_to_idx=doc_id_to_idx
)


**Truy vấn:** 'cướp tiệm vàng'
-> Bộ lọc Mục lục tìm thấy: 13391 văn bản ứng viên (Thay vì phải quét 160254 bài).

--- Top 5 Tài liệu tương đồng nhất ---

Rank 1 (Similarity Score: 0.6552):
  ID: 217072 | Title: Vụ cướp tiệm vàng tại chợ Đông Ba: Công an yêu cầu người dân trả lại vàng đã nhặt
  Content snippet: vụ cướp tiệm vàng tại chợ đông_ba_công an yêu_cầu người_dân trả lại vàng đã nhặt sau khi bắt được nghi phạm trong vụ nổ_súng cướp tiệm vàng tại chợ đô...

Rank 2 (Similarity Score: 0.6498):
  ID: 217762 | Title: Công an đã bắt kẻ dùng súng cướp tiệm vàng ở Huế
  Content snippet: công_an đã bắt kẻ dùng súng cướp tiệm vàng ở huế chiều date0731 lãnh_đạo công_an tỉnh thừa_thiên_huế cho biết hiện cơ_quan công_an đã bắt được đối_tượ...

Rank 3 (Similarity Score: 0.6328):
  ID: 217737 | Title: Nhân chứng kể lại giây phút đối tượng nổ súng cướp tiệm vàng ở TP Huế
  Content snippet: nhân_chứng kể lại giây_phút đối_tượng nổ_súng cướp tiệm vàng ở tp huế qua trao_đổi bà h người có cửa_hàng

In [6]:
# Lưu mô hình vào ổ cứng local (thư mục gốc của project)
save_dir = 'models/'
os.makedirs(save_dir, exist_ok=True)

vectorizer_path = os.path.join(save_dir, 'tfidf_vectorizer.pkl')
tfidf_matrix_path = os.path.join(save_dir, 'tfidf_matrix.npz')

# 1. Lưu TfidfVectorizer
with open(vectorizer_path, 'wb') as f:
    pickle.dump(vectorizer, f)
print(f"TfidfVectorizer đã lưu thành công tại: {vectorizer_path}")

# 2. Lưu ma trận Sparse TF-IDF (Dùng save_npz sẽ tiết kiệm dung lượng rất nhiều)
save_npz(tfidf_matrix_path, tfidf_matrix)
print(f"Ma trận TF-IDF đã lưu thành công tại: {tfidf_matrix_path}")

TfidfVectorizer đã lưu thành công tại: models/tfidf_vectorizer.pkl
Ma trận TF-IDF đã lưu thành công tại: models/tfidf_matrix.npz
